# BÀI TẬP LỚN — NỀN TẢNG LẬP TRÌNH CHO PHÂN TÍCH VÀ TRỰC QUAN DỮ LIỆU
**Học kỳ 261 – Năm học 2026–2027 · Giảng viên: Lê Thành Sách**

## BÀI 1 — DỮ LIỆU BẢNG (TABULAR): DỰ ĐOÁN KHÁCH HÀNG RỜI BỎ NGÂN HÀNG
- **Nhóm:** `<Tên nhóm>` — **Thành viên:** `<Họ tên – MSSV – vai trò>`

**Bài toán:** phân loại nhị phân — dự đoán khách hàng có rời bỏ ngân hàng (`Exited = 1`) hay không,
để bộ phận chăm sóc khách hàng ưu tiên giữ chân đúng người.

**Câu hỏi chính của bài:**
1. Những đặc điểm nào phân biệt khách hàng rời bỏ với khách hàng ở lại?
2. Dữ liệu mất cân bằng (~20% rời bỏ) ảnh hưởng thế nào và **xử lý mất cân bằng giúp bắt thêm được bao nhiêu khách hàng rời bỏ**?

**Quy trình:** EDA → làm sạch & tiền xử lý trong Pipeline (chống rò rỉ dữ liệu) → so sánh baseline với 3 mô hình →
tinh chỉnh siêu tham số và ngưỡng quyết định → đánh giá một lần trên tập Test → phân tích lỗi và suy luận.

## 0. CÀI ĐẶT MÔI TRƯỜNG

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve)
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
CLASS_COLORS = {0: '#4C72B0', 1: '#DD5145'}
CLASS_NAMES = {0: 'Ở lại (0)', 1: 'Rời bỏ (1)'}
print('Môi trường đã sẵn sàng!')

## 1. MÔ TẢ TẬP DỮ LIỆU VÀ KHÁM PHÁ (EDA)

**Nguồn:** *Bank Customer Churn* (Churn_Modelling) — 10,000 khách hàng của một ngân hàng châu Âu (Pháp, Đức, Tây Ban Nha), 14 cột.

> ⚠️ **Lưu ý minh bạch:** dữ liệu gốc **không có missing values**. Để thực hành thống kê và xử lý thiếu dữ liệu theo yêu cầu đề bài,
> nhóm tạo missing **ngẫu nhiên hoàn toàn (MCAR)** ở 5 cột với tỉ lệ 2–4% bằng hàm `inject_missing` (seed = 42, script
> `scripts/prepare_churn_dataset.py`). Dữ liệu đọc từ file local hay tải từ URL gốc đều cho kết quả giống hệt nhau.

### 1.1. Tải dữ liệu

In [ ]:
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/perckgg/NTLT/main/01_Tabular/data/tabular'
ORIGINAL_URL = 'https://raw.githubusercontent.com/selva86/datasets/master/Churn_Modelling.csv'
DATA_FILE = 'customer_churn_dataset.csv'
LOCAL_DIRS = ['data/tabular', '../data/tabular', '/content/data/tabular', '/content']
MISSING_RATES = {'CreditScore': 0.025, 'Geography': 0.020, 'Age': 0.035, 'Balance': 0.030, 'EstimatedSalary': 0.040}


def inject_missing(df, rates=MISSING_RATES, seed=RANDOM_STATE):
    """Tạo missing MCAR — giống hệt scripts/prepare_churn_dataset.py."""
    df = df.copy()
    rng = np.random.default_rng(seed)
    for col, rate in rates.items():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(float)
        df.loc[rng.random(len(df)) < rate, col] = np.nan
    return df


def load_data():
    for d in LOCAL_DIRS:
        if (Path(d) / DATA_FILE).exists():
            return pd.read_csv(Path(d) / DATA_FILE), f'local: {Path(d) / DATA_FILE}'
    try:
        return pd.read_csv(f'{GITHUB_RAW_BASE}/{DATA_FILE}'), 'GitHub của nhóm'
    except Exception:
        return inject_missing(pd.read_csv(ORIGINAL_URL)), 'URL gốc + inject_missing(seed=42)'


df, source = load_data()
print(f'Nguồn: {source} | {df.shape[0]:,} dòng × {df.shape[1]} cột')
df.head()

### 1.2. Tổng quan và mô tả thuộc tính

| Cột | Kiểu | Mô tả |
|---|---|---|
| RowNumber, CustomerId, Surname | Định danh | Chỉ để quản lý — **loại bỏ** khi mô hình hóa |
| CreditScore | Numerical | Điểm tín dụng (350–850) |
| Geography | Categorical | Quốc gia: France / Germany / Spain |
| Gender | Categorical | Giới tính |
| Age | Numerical | Tuổi |
| Tenure | Numerical | Số năm là khách hàng (0–10) |
| Balance | Numerical | Số dư tài khoản |
| NumOfProducts | Numerical (rời rạc) | Số sản phẩm đang dùng (1–4) |
| HasCrCard | Binary | Có thẻ tín dụng |
| IsActiveMember | Binary | Có hoạt động thường xuyên |
| EstimatedSalary | Numerical | Lương ước tính |
| **Exited** | **Target** | 1 = rời bỏ, 0 = ở lại |

In [ ]:
ID_COLS = ['RowNumber', 'CustomerId', 'Surname']
TARGET = 'Exited'
NUM_COLS = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
BIN_COLS = ['HasCrCard', 'IsActiveMember']
CAT_COLS = ['Geography', 'Gender']

overview = pd.Series({'Số mẫu': len(df), 'Số feature dùng được': len(NUM_COLS + BIN_COLS + CAT_COLS),
                      'Numerical': len(NUM_COLS), 'Binary': len(BIN_COLS), 'Categorical': len(CAT_COLS),
                      'Dòng trùng lặp': df.duplicated().sum(), 'CustomerId trùng': df['CustomerId'].duplicated().sum()})
display(overview.to_frame('Giá trị'))
df.info()

for label in [0, 1]:
    print(f'\nMẫu dữ liệu — {CLASS_NAMES[label]}:')
    display(df[df[TARGET] == label].drop(columns=ID_COLS).head(3))

### 1.3. Phân tích Missing Values
- **What:** cột nào thiếu, thiếu bao nhiêu. **Why:** missing làm lệch thống kê và nhiều mô hình không chấp nhận NaN.
- **How (quy tắc trong báo cáo mẫu):** < 5% → điền đơn giản (median/mode); 5–30% → phương pháp nâng cao; > 50% → cân nhắc bỏ cột.

In [ ]:
miss = df.isna().sum()
miss = pd.DataFrame({'Missing': miss, 'Tỉ lệ (%)': miss / len(df) * 100})
miss = miss[miss['Missing'] > 0].sort_values('Missing', ascending=False)
miss['Mức độ'] = pd.cut(miss['Tỉ lệ (%)'], [0, 5, 30, 100], labels=['Thấp', 'Trung bình', 'Cao'])
print(f'Tổng ô thiếu: {miss.Missing.sum():,} | số cột thiếu: {len(miss)} | dòng có ít nhất 1 ô thiếu: {df.isna().any(axis=1).mean():.1%}')
display(miss)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(x=miss['Tỉ lệ (%)'], y=miss.index, ax=axes[0], color='#DD5145')
axes[0].axvline(5, ls='--', c='gray'); axes[0].set_title('Tỉ lệ missing theo cột (ngưỡng 5%)')
rate_by_class = df[miss.index].isna().groupby(df[TARGET]).mean().T * 100
rate_by_class.columns = [CLASS_NAMES[c] for c in rate_by_class.columns]
rate_by_class.plot.bar(ax=axes[1], color=list(CLASS_COLORS.values()), rot=0)
axes[1].set(title='Tỉ lệ missing theo lớp — kiểm tra missing có phụ thuộc nhãn không', ylabel='%')
plt.tight_layout(); plt.show()

### 1.4. Phân phối các biến Numerical
- **Why:** hình dạng phân phối quyết định cách scaling/biến đổi; boxplot cho thấy outlier.
- **How:** so sánh mean với median, xem độ lệch (skewness); |skew| > 1 là lệch mạnh.

In [ ]:
stats = df[NUM_COLS].describe().T[['mean', '50%', 'std', 'min', 'max']].rename(columns={'50%': 'median'})
stats['skew'] = df[NUM_COLS].skew()
stats['% bằng 0'] = (df[NUM_COLS] == 0).mean() * 100
display(stats)

fig, axes = plt.subplots(2, len(NUM_COLS), figsize=(22, 7), gridspec_kw={'height_ratios': [3, 1]})
for i, col in enumerate(NUM_COLS):
    sns.histplot(df[col].dropna(), bins=30, kde=col not in ['NumOfProducts', 'Tenure'], ax=axes[0, i], color='#4C72B0')
    axes[0, i].set_title(col)
    sns.boxplot(x=df[col], ax=axes[1, i], color='#8DA0CB')
plt.suptitle('Phân phối (trên) và boxplot (dưới) của các biến số', y=1.02)
plt.tight_layout(); plt.show()

### 1.5. Phân phối các biến Categorical / Binary
- **How (báo cáo mẫu):** cardinality < 10 → One-Hot Encoding; > 50 → Target/Frequency Encoding; nhóm hiếm (< 1%) nên gộp.

In [ ]:
cat_info = pd.DataFrame({c: {'Số giá trị': df[c].nunique(), 'Mode': df[c].mode()[0],
                             'Tỉ lệ mode': df[c].value_counts(normalize=True).iloc[0],
                             'Nhóm nhỏ nhất': df[c].value_counts(normalize=True).iloc[-1]}
                         for c in CAT_COLS + BIN_COLS}).T
display(cat_info)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, CAT_COLS + BIN_COLS):
    sns.countplot(data=df, x=col, ax=ax, color='#55A868', order=df[col].value_counts().index)
    ax.bar_label(ax.containers[0], fmt='%d')
    ax.set_title(col)
plt.tight_layout(); plt.show()

### 1.6. Phân phối biến mục tiêu
- **How:** cân bằng ~ 40:60–50:50; mất cân bằng > 2:1; rất mất cân bằng > 10:1. Mất cân bằng → dùng stratified split,
  class weight và các metric F1 / Recall / ROC-AUC / PR-AUC thay cho Accuracy.

In [ ]:
counts = df[TARGET].value_counts().sort_index()
print(f'Ở lại: {counts[0]:,} ({counts[0] / len(df):.1%}) | Rời bỏ: {counts[1]:,} ({counts[1] / len(df):.1%}) '
      f'| tỉ lệ {counts[0] / counts[1]:.1f}:1')
print(f'→ Một mô hình luôn đoán "Ở lại" đã đạt Accuracy {counts[0] / len(df):.1%} nhưng không bắt được khách hàng rời bỏ nào.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].pie(counts, labels=[CLASS_NAMES[i] for i in counts.index], autopct='%1.1f%%',
            colors=list(CLASS_COLORS.values()), startangle=90)
axes[0].set_title('Tỉ lệ các lớp')
bars = axes[1].bar([CLASS_NAMES[i] for i in counts.index], counts, color=list(CLASS_COLORS.values()))
axes[1].bar_label(bars); axes[1].set_title('Số lượng mỗi lớp')
plt.tight_layout(); plt.show()

### 1.7. Phân tích tương quan
- **How:** |r| > 0.7 → đa cộng tuyến, cân nhắc bỏ bớt một biến; |r| < 0.3 → tương quan tuyến tính yếu.

In [ ]:
corr = df[NUM_COLS + BIN_COLS + [TARGET]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, mask=np.triu(np.ones_like(corr, dtype=bool), 1))
plt.title('Ma trận tương quan Pearson'); plt.show()

pairs = corr.drop(index=TARGET, columns=TARGET).where(np.triu(np.ones((len(corr) - 1,) * 2, dtype=bool), 1)).stack()
print(f'Cặp feature tương quan mạnh nhất: {pairs.abs().idxmax()} r = {pairs[pairs.abs().idxmax()]:.2f}')
print('Tương quan với Exited:', corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(2).to_dict())

### 1.8. Phát hiện Outlier (phương pháp IQR)
Giá trị ngoài khoảng $[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$ được coi là outlier. Mức độ: < 1% thấp, 1–5% trung bình, > 5% cao.

In [ ]:
def iqr_table(data, cols, k=1.5):
    rows = []
    for c in cols:
        s = data[c].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        lo, hi = q1 - k * (q3 - q1), q3 + k * (q3 - q1)
        n = ((s < lo) | (s > hi)).sum()
        rows.append({'Feature': c, 'Outliers': n, 'Tỉ lệ (%)': n / len(s) * 100,
                     'Khoảng hợp lệ': f'[{lo:,.1f}, {hi:,.1f}]'})
    t = pd.DataFrame(rows).set_index('Feature')
    t['Mức độ'] = pd.cut(t['Tỉ lệ (%)'], [-0.01, 1, 5, 100], labels=['Thấp', 'Trung bình', 'Cao'])
    return t


outliers = iqr_table(df, NUM_COLS)
display(outliers)
age_out = df['Age'] > df['Age'].quantile(0.75) + 1.5 * (df['Age'].quantile(0.75) - df['Age'].quantile(0.25))
print(f'Tỉ lệ rời bỏ ở nhóm outlier Age (> 62 tuổi, n={age_out.sum()}): {df.loc[age_out, TARGET].mean():.1%} '
      f'(toàn bộ: {df[TARGET].mean():.1%})')
display(df.groupby(pd.cut(df['Age'], [17, 30, 40, 50, 62, 70, 100]), observed=True)[TARGET]
          .agg(['size', 'mean']).rename(columns={'size': 'Số KH', 'mean': '% rời bỏ'}).rename_axis('Nhóm tuổi'))

### 1.9. Quan hệ giữa biến mục tiêu và các feature
- **How:** với categorical so sánh **tỉ lệ rời bỏ** theo từng nhóm; với numerical so sánh phân phối theo lớp.
  Chênh lệch lớn → feature có khả năng dự báo tốt.

In [ ]:
overall = df[TARGET].mean()
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, ['Geography', 'Gender', 'NumOfProducts', 'IsActiveMember']):
    rate = df.groupby(col)[TARGET].agg(['mean', 'size'])
    sns.barplot(x=rate.index.astype(str), y=rate['mean'] * 100, ax=ax, color='#DD5145')
    ax.bar_label(ax.containers[0], labels=[f'{m:.0%}\n(n={n:,})' for m, n in rate.values], fontsize=9)
    ax.axhline(overall * 100, ls='--', c='gray', label=f'Trung bình {overall:.0%}')
    ax.set(title=f'Tỉ lệ rời bỏ theo {col}', ylabel='% rời bỏ', xlabel='', ylim=(0, rate['mean'].max() * 125))
    ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ['Age', 'Balance', 'CreditScore']):
    sns.kdeplot(data=df.assign(Lớp=df[TARGET].map(CLASS_NAMES)), x=col, hue='Lớp', common_norm=False, fill=True,
                palette={CLASS_NAMES[k]: v for k, v in CLASS_COLORS.items()}, ax=ax)
    ax.set_title(f'{col} theo lớp')
plt.tight_layout(); plt.show()
display(df.groupby(TARGET)[NUM_COLS].median().rename(index=CLASS_NAMES).T.rename_axis('Median'))

### 1.10. Key Insights từ EDA


**Dữ liệu và chất lượng**
- 10,000 khách hàng × 14 cột, không có dòng trùng lặp hay CustomerId trùng. Loại 3 cột định danh, còn 10 feature: 6 numerical, 2 binary, 2 categorical.
- **Missing:** 1,521 ô thiếu ở 5 cột, mỗi cột 2.1–3.9% (đều ở mức *Thấp* < 5%), nhưng **14.1% số dòng** có ít nhất 1 ô thiếu.
  Xóa dòng sẽ mất 14% dữ liệu, nên chọn **điền median (numerical) / mode (categorical)**.
  Tỉ lệ thiếu ở hai lớp gần như bằng nhau (ví dụ Age: 3.8% so với 3.5%), đúng với cơ chế MCAR đã tạo.

**Phân phối**
- **Age** lệch phải (skew = 1.01; median 37, max 92). **Balance** có hai đỉnh: **35.2% khách hàng có số dư bằng 0**,
  nên mean (76k) nhỏ hơn hẳn median (97k). Đây là lý do tạo thêm feature `IsZeroBalance`.
- CreditScore và EstimatedSalary gần đối xứng (|skew| < 0.1); EstimatedSalary gần như phân phối đều.
- Các biến categorical có cardinality ≤ 3, nên dùng **One-Hot Encoding** theo quy tắc của báo cáo mẫu.

**Mục tiêu và tương quan**
- **Mất cân bằng 3.9 : 1** (79.6% ở lại, 20.4% rời bỏ): mô hình luôn đoán "ở lại" đã đạt Accuracy 79.6%.
  Vì vậy cần chia tập phân tầng, dùng class weight, và đánh giá bằng F1 / Recall / PR-AUC.
- Không có đa cộng tuyến: cặp tương quan mạnh nhất chỉ là Balance–NumOfProducts (r = −0.30).
  Tương quan tuyến tính với Exited đều yếu (Age 0.29, IsActiveMember −0.16, Balance 0.12), nên cần mô hình phi tuyến.
- **Outlier:** chủ yếu ở Age (348 KH > 62 tuổi, 3.6%); NumOfProducts có 60 KH dùng 4 sản phẩm (0.6%).

**Quan hệ target – feature (yếu tố phân biệt rõ nhất)**
- **NumOfProducts quan hệ phi tuyến:** 1 sản phẩm 27.7%, **2 sản phẩm chỉ 7.6%**, 3 sản phẩm 82.7%, 4 sản phẩm **100%** rời bỏ.
- **Age:** median 45 tuổi (rời bỏ) so với 36 (ở lại). Tỉ lệ rời bỏ tăng từ 7.5% (≤ 30 tuổi) lên **54.7% (50–62 tuổi)** rồi giảm còn 8.7% (> 70 tuổi).
- **Germany 32.4%** so với France 16.1% / Spain 16.6%; **nữ 25.1%** so với nam 16.5%; **không hoạt động 26.9%** so với hoạt động 14.3%.

## 2. CHUẨN BỊ DỮ LIỆU

Nguyên tắc: mọi bước **có học tham số từ dữ liệu** (median để điền thiếu, IQR để scale, danh mục để one-hot) đều nằm trong
`Pipeline` và chỉ được `fit` trên tập **Train** → tránh rò rỉ dữ liệu (data leakage) sang Validation/Test.

### 2.1. Loại cột định danh và kiểm tra trùng lặp

In [ ]:
data = df.drop(columns=ID_COLS)
print(f'Sau khi bỏ {ID_COLS}: {data.shape[1]} cột | dòng trùng lặp: {data.duplicated().sum()}')

### 2.2. Tạo đặc trưng (Feature Engineering) — dựa trên EDA
| Đặc trưng | Lý do (từ EDA) |
|---|---|
| `IsZeroBalance` = 1 nếu `Balance == 0` | Balance có phân phối **hai đỉnh**: ~36% khách hàng số dư bằng 0, nhóm này có tỉ lệ rời bỏ khác hẳn |
| `ProductsGroup` = `NumOfProducts` dạng **categorical**, gộp 3 và 4 thành `3+` | Tỉ lệ rời bỏ theo số sản phẩm **không tuyến tính** (2 sản phẩm thấp nhất, ≥ 3 rất cao); nhóm 4 hiếm (< 1%) nên gộp theo quy tắc nhóm hiếm |

Các phép biến đổi này chỉ tính trên **từng dòng** (không học tham số), được đóng gói bằng `FunctionTransformer`
để hàm suy luận nhận trực tiếp dữ liệu thô.

In [ ]:
def add_features(X):
    X = X.copy()
    X['IsZeroBalance'] = np.where(X['Balance'].isna(), np.nan, (X['Balance'] == 0).astype(float))
    X['ProductsGroup'] = X['NumOfProducts'].clip(upper=3).astype(int).astype(str).replace({'3': '3+'})
    return X.drop(columns=['NumOfProducts'])


feature_maker = FunctionTransformer(add_features)
print(add_features(data.drop(columns=TARGET)).head())
print('\nTỉ lệ rời bỏ theo IsZeroBalance:',
      data.groupby(add_features(data)['IsZeroBalance'])[TARGET].mean().round(3).to_dict())

### 2.3. Chia tập Train / Validation / Test (70 / 15 / 15, phân tầng theo nhãn)
- **Train:** huấn luyện. **Validation:** so sánh mô hình, chọn ngưỡng. **Test:** chỉ dùng **một lần** ở Phần 4.
- `stratify=y` giữ tỉ lệ rời bỏ ~20% ở cả ba tập. Chia **trước** khi điền missing/scale để tránh rò rỉ.

In [ ]:
X, y = data.drop(columns=TARGET), data[TARGET]
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)

display(pd.DataFrame({name: {'Số mẫu': len(t), 'Tỉ lệ': len(t) / len(X), 'Số rời bỏ': int(t.sum()), '% rời bỏ': t.mean()}
                      for name, t in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]}).T)

### 2.4. Pipeline tiền xử lý
| Nhóm cột | Missing | Outlier / Scaling | Encoding |
|---|---|---|---|
| Numerical (`CreditScore, Age, Tenure, Balance, EstimatedSalary`) | median | **RobustScaler** (dựa trên median/IQR, ít bị outlier kéo lệch) | — |
| Binary (`HasCrCard, IsActiveMember, IsZeroBalance`) | mode | giữ 0/1 | — |
| Categorical (`Geography, Gender, ProductsGroup`) | mode | — | **One-Hot** (cardinality ≤ 3) |

**Xử lý outlier:** outlier của `Age` (63–92 tuổi) là **giá trị thật**, không phải lỗi nhập liệu. Tỉ lệ rời bỏ tăng theo tuổi đến
khoảng 60 rồi **giảm trở lại** ở nhóm này (xem 1.8 và 1.9); xóa đi sẽ làm mất quy luật phi tuyến đó. Vì vậy nhóm **giữ lại**
outlier và dùng RobustScaler để giảm ảnh hưởng tới mô hình tuyến tính; mô hình cây vốn không nhạy với outlier.

In [ ]:
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']
BIN_FEATURES = ['HasCrCard', 'IsActiveMember', 'IsZeroBalance']
CAT_FEATURES = ['Geography', 'Gender', 'ProductsGroup']


def build_preprocessor():
    columns = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())]), NUM_FEATURES),
        ('bin', SimpleImputer(strategy='most_frequent'), BIN_FEATURES),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT_FEATURES),
    ], verbose_feature_names_out=False)
    return Pipeline([('features', FunctionTransformer(add_features)), ('columns', columns)])


prep = build_preprocessor().fit(X_train)
Xt_train = pd.DataFrame(prep.transform(X_train), columns=prep.named_steps['columns'].get_feature_names_out(),
                        index=X_train.index)
print(f'Sau tiền xử lý: {Xt_train.shape[1]} cột | missing còn lại: {int(Xt_train.isna().sum().sum())}')
print('Median dùng để điền (học từ Train):',
      {c: float(v) for c, v in zip(NUM_FEATURES, prep.named_steps['columns'].named_transformers_['num']['imputer'].statistics_.round(1))})
Xt_train.head()

### 2.5. Kiểm tra trực quan: trước và sau khi điền thiếu và scale
So sánh phân phối gốc với hai cách scale cho hai biến có outlier/lệch: `Age` và `Balance`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for r, col in enumerate(['Age', 'Balance']):
    raw = X_train[col]
    filled = raw.fillna(raw.median())
    std_scaled = StandardScaler().fit_transform(filled.to_frame()).ravel()
    sns.histplot(raw.dropna(), bins=40, ax=axes[r, 0], color='#4C72B0')
    axes[r, 0].set_title(f'{col} — gốc (thiếu {raw.isna().sum()} giá trị)')
    sns.histplot(std_scaled, bins=40, ax=axes[r, 1], color='#8DA0CB')
    axes[r, 1].set_title(f'StandardScaler — khoảng [{std_scaled.min():.1f}, {std_scaled.max():.1f}]')
    sns.histplot(Xt_train[col], bins=40, ax=axes[r, 2], color='#55A868')
    axes[r, 2].set_title(f'RobustScaler (dùng trong pipeline) — IQR = 1, median = 0')
plt.tight_layout(); plt.show()

**Nhận xét Phần 2:**
- `IsZeroBalance` tách được hai nhóm có tỉ lệ rời bỏ khác nhau: số dư bằng 0 rời bỏ **13.8%**, có số dư rời bỏ **24.0%**.
- Sau khi chia phân tầng, tỉ lệ rời bỏ ở Train / Validation / Test lần lượt là 20.4% / 20.4% / 20.3%, tức ba tập đồng nhất.
- Pipeline biến 10 cột gốc thành 16 cột số, không còn missing. Median để điền thiếu chỉ học từ Train (ví dụ Age = 37).
- Hình trên cho thấy StandardScaler bị outlier Age kéo dài đuôi phải, còn RobustScaler giữ phần thân phân phối quanh 0 với IQR = 1.

## 3. PHÂN TÍCH VÀ MÔ HÌNH HÓA

### 3.1. Thước đo đánh giá
Vì dữ liệu mất cân bằng (~20% rời bỏ), Accuracy dễ gây hiểu nhầm. Nhóm dùng:
- **F1 (lớp rời bỏ)** — thước đo chính, cân bằng giữa Precision và Recall.
- **Recall** — tỉ lệ khách hàng rời bỏ bắt được (bỏ sót một khách hàng sắp rời đi thường tốn kém hơn gọi nhầm một khách hàng ở lại).
- **ROC-AUC, PR-AUC** — đánh giá khả năng xếp hạng, không phụ thuộc ngưỡng.

### 3.2. Baseline và so sánh các mô hình (trên Validation)
- **Baseline 1 — Dummy:** luôn đoán lớp đa số.
- **Baseline 2 — Logistic Regression** không xử lý mất cân bằng.
- **Mô hình mở rộng:** Logistic Regression, Random Forest, HistGradientBoosting — mỗi mô hình chạy **hai phiên bản**:
  không trọng số và `class_weight='balanced'` (phạt lỗi trên lớp thiểu số nặng hơn).

In [ ]:
def evaluate(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {'Accuracy': accuracy_score(y_true, pred), 'Precision': precision_score(y_true, pred, zero_division=0),
            'Recall': recall_score(y_true, pred), 'F1': f1_score(y_true, pred),
            'ROC-AUC': roc_auc_score(y_true, prob), 'PR-AUC': average_precision_score(y_true, prob)}


def make_pipeline(clf):
    return Pipeline([('prep', build_preprocessor()), ('clf', clf)])


ALGORITHMS = {
    'LogisticRegression': lambda cw: LogisticRegression(max_iter=2000, class_weight=cw),
    'RandomForest': lambda cw: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight=cw,
                                                      n_jobs=-1, random_state=RANDOM_STATE),
    'HistGradientBoosting': lambda cw: HistGradientBoostingClassifier(class_weight=cw, random_state=RANDOM_STATE),
}

models = {'Dummy (lớp đa số)': make_pipeline(DummyClassifier(strategy='most_frequent'))}
for name, factory in ALGORITHMS.items():
    models[name] = make_pipeline(factory(None))
    models[f'{name} + balanced'] = make_pipeline(factory('balanced'))

val_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    val_rows.append({'Mô hình': name, **evaluate(y_val, model.predict_proba(X_val)[:, 1])})
val_results = pd.DataFrame(val_rows).set_index('Mô hình')
display(val_results.style.format('{:.3f}').highlight_max(color='#c6efce', axis=0))

**Câu hỏi: xử lý mất cân bằng (`class_weight='balanced'`) thay đổi Recall và F1 của từng thuật toán thế nào?**

In [ ]:
plot_rows = []
for name in ALGORITHMS:
    for suffix, label in [('', 'Không trọng số'), (' + balanced', 'class_weight=balanced')]:
        r = val_results.loc[name + suffix]
        plot_rows += [{'Thuật toán': name, 'Phiên bản': label, 'Metric': m, 'Giá trị': r[m]} for m in ['Recall', 'F1']]
plot_df = pd.DataFrame(plot_rows)

g = sns.catplot(data=plot_df, x='Thuật toán', y='Giá trị', hue='Phiên bản', col='Metric', kind='bar',
                palette=['#BBBBBB', '#DD5145'], height=4, aspect=1.4)
for ax in g.axes.flat:
    for c in ax.containers:
        ax.bar_label(c, fmt='%.2f', fontsize=9)
    ax.set_ylim(0, 1)
g.figure.suptitle('Validation: ảnh hưởng của class_weight lên Recall và F1 (lớp rời bỏ)', y=1.05)
plt.show()

**Nhận xét Validation:**
- **Accuracy gây hiểu nhầm:** Dummy đạt Accuracy 0.796 nhưng F1 = 0. Logistic Regression thường có Accuracy 0.844 nhưng **chỉ bắt được 36% khách hàng rời bỏ**.
- **`class_weight='balanced'` tăng mạnh Recall ở cả 3 thuật toán:** LogReg 0.36 → **0.79**, RandomForest 0.49 → 0.65, HistGB 0.48 → 0.72.
  F1 cũng tăng (0.487 → 0.583; 0.613 → 0.642; 0.590 → 0.630). Đổi lại Precision giảm (ví dụ LogReg 0.74 → 0.46).
- **ROC-AUC gần như không đổi** khi thêm trọng số (ví dụ RF 0.881 so với 0.879): class weight không làm mô hình phân biệt tốt hơn,
  mà chủ yếu **dịch ranh giới quyết định** về phía lớp thiểu số.
- Mô hình cây vượt Logistic Regression (ROC-AUC ≈ 0.88 so với 0.85; PR-AUC 0.72 so với 0.66), khớp với EDA:
  quan hệ giữa Age, NumOfProducts và Exited là phi tuyến.

### 3.3. Tinh chỉnh siêu tham số (GridSearchCV, 5-fold phân tầng trên Train)
Tinh chỉnh hai mô hình cây tốt nhất, coi `class_weight` cũng là một siêu tham số; tiêu chí chọn là **F1**.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
GRIDS = {
    'RandomForest': (ALGORITHMS['RandomForest'](None),
                     {'clf__max_depth': [8, 12, None], 'clf__min_samples_leaf': [2, 5, 10],
                      'clf__class_weight': [None, 'balanced']}),
    'HistGradientBoosting': (ALGORITHMS['HistGradientBoosting'](None),
                             {'clf__learning_rate': [0.05, 0.1], 'clf__max_leaf_nodes': [15, 31],
                              'clf__max_iter': [100, 300], 'clf__class_weight': [None, 'balanced']}),
}

tuned, tune_rows = {}, []
for name, (clf, grid) in GRIDS.items():
    search = GridSearchCV(make_pipeline(clf), grid, scoring='f1', cv=cv, n_jobs=-1).fit(X_train, y_train)
    tuned[name] = search.best_estimator_
    prob = tuned[name].predict_proba(X_val)[:, 1]
    tune_rows.append({'Mô hình': f'{name} (tuned)', 'CV F1': search.best_score_, **evaluate(y_val, prob),
                      'Tham số tốt nhất': {k.replace('clf__', ''): v for k, v in search.best_params_.items()}})
tune_results = pd.DataFrame(tune_rows).set_index('Mô hình')
display(tune_results.drop(columns='Tham số tốt nhất'))
for name, params in tune_results['Tham số tốt nhất'].items():
    print(f'{name}: {params}')

### 3.4. Chọn ngưỡng quyết định trên Validation
Mặc định mô hình dự đoán "rời bỏ" khi xác suất ≥ 0.5. Với dữ liệu mất cân bằng, ngưỡng này thường không tối ưu.
Nhóm quét ngưỡng 0.10–0.90 **trên Validation** và chọn ngưỡng cho F1 cao nhất. Tập Test không tham gia bước này.

In [ ]:
THRESHOLDS = np.round(np.arange(0.10, 0.91, 0.01), 2)
best_name = tune_results['F1'].idxmax().replace(' (tuned)', '')
final_model = tuned[best_name]
val_prob = final_model.predict_proba(X_val)[:, 1]

curve = pd.DataFrame([{'threshold': t, **evaluate(y_val, val_prob, t)} for t in THRESHOLDS]).set_index('threshold')
best_threshold = float(curve['F1'].idxmax())
print(f'Mô hình cuối: {best_name} | ngưỡng tối ưu trên Validation = {best_threshold:.2f}')
display(curve.loc[[0.5, best_threshold], ['Precision', 'Recall', 'F1']].rename(index=lambda t: f'ngưỡng {t:.2f}'))

ax = curve[['Precision', 'Recall', 'F1']].plot(figsize=(10, 4), color=['#4C72B0', '#DD5145', '#55A868'])
ax.axvline(0.5, ls=':', c='gray', label='0.5 (mặc định)')
ax.axvline(best_threshold, ls='--', c='k', label=f'{best_threshold:.2f} (tối ưu F1)')
ax.set(title=f'{best_name}: Precision / Recall / F1 theo ngưỡng (Validation)', xlabel='Ngưỡng', ylabel='Giá trị')
ax.legend(); plt.show()

**Nhận xét tinh chỉnh:**
- Khi `class_weight` được đưa vào lưới tìm kiếm, **cả hai mô hình đều chọn `balanced`**. Đây là bằng chứng độc lập (qua 5-fold CV)
  rằng xử lý mất cân bằng có lợi cho F1.
- RandomForest và HistGradientBoosting sau tinh chỉnh gần như ngang nhau (F1 Validation 0.634 so với 0.630, PR-AUC đều 0.718).
  Nhóm chọn **RandomForest** vì F1 cao hơn một chút.
- **Chọn ngưỡng:** ở ngưỡng 0.5 mô hình đạt Recall 0.716 nhưng Precision chỉ 0.569. Ngưỡng **0.61** cho F1 cao nhất (0.646),
  với Precision 0.693 và Recall 0.605, tức giảm đáng kể số lần báo nhầm và chỉ mất một phần Recall.

## 4. TỔNG KẾT VÀ MỞ RỘNG

### 4.1. Đánh giá trên tập Test độc lập
Mô hình, siêu tham số và ngưỡng đều đã được chốt ở Phần 3; tập Test chỉ được dùng **một lần** ở đây.

In [ ]:
test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)
print(f'=== {best_name} (tuned), ngưỡng {best_threshold:.2f} — TẬP TEST ===')
print(classification_report(y_test, test_pred, target_names=list(CLASS_NAMES.values()), digits=3))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, test_pred), display_labels=list(CLASS_NAMES.values())).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion matrix (ngưỡng {best_threshold:.2f})')
for name, model in {'LogisticRegression (baseline)': models['LogisticRegression'], **tuned}.items():
    p = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p)
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, p):.3f})')
    axes[2].plot(rec, prec, label=f'{name} (AP={average_precision_score(y_test, p):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(title='ROC curve (Test)', xlabel='False Positive Rate', ylabel='True Positive Rate')
axes[2].axhline(y_test.mean(), ls='--', c='gray', label=f'Ngẫu nhiên ({y_test.mean():.2f})')
axes[2].set(title='Precision–Recall curve (Test)', xlabel='Recall', ylabel='Precision')
for ax in axes[1:]:
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 4.2. Câu hỏi: *Từng bước cải tiến giúp bắt thêm được bao nhiêu khách hàng rời bỏ trên tập Test?*
So sánh 4 cấu hình trên **cùng tập Test**:
1. **Baseline** — Logistic Regression, không xử lý mất cân bằng, ngưỡng 0.5.
2. **Mô hình tốt nhất, không trọng số** — cùng siêu tham số với mô hình cuối nhưng `class_weight=None`, ngưỡng 0.5.
3. **+ class_weight='balanced'**, ngưỡng 0.5.
4. **+ ngưỡng tối ưu** chọn trên Validation (mô hình cuối).

In [ ]:
unweighted = clone(final_model).set_params(clf__class_weight=None).fit(X_train, y_train)
balanced = clone(final_model).set_params(clf__class_weight='balanced').fit(X_train, y_train)
balanced_val_prob = balanced.predict_proba(X_val)[:, 1]
balanced_threshold = max(THRESHOLDS, key=lambda t: f1_score(y_val, (balanced_val_prob >= t).astype(int)))

steps = [('1. Baseline LogReg', models['LogisticRegression'], 0.5),
         (f'2. {best_name}', unweighted, 0.5),
         ('3. + class_weight', balanced, 0.5),
         (f'4. + class_weight + ngưỡng {balanced_threshold:.2f}', balanced, balanced_threshold)]
step_rows = []
for label, model, thr in steps:
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= thr).astype(int)
    step_rows.append({'Cấu hình': label, **evaluate(y_test, prob, thr),
                      'Bắt được (TP)': int(((pred == 1) & (y_test == 1)).sum()),
                      'Báo nhầm (FP)': int(((pred == 1) & (y_test == 0)).sum())})
steps_df = pd.DataFrame(step_rows).set_index('Cấu hình')
display(steps_df.style.format({c: '{:.3f}' for c in ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']}))

fig, ax = plt.subplots(figsize=(13, 5))
long = steps_df[['Precision', 'Recall', 'F1']].reset_index().melt('Cấu hình', var_name='Metric', value_name='Giá trị')
sns.barplot(data=long, x='Cấu hình', y='Giá trị', hue='Metric', palette=['#4C72B0', '#DD5145', '#55A868'], ax=ax)
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', fontsize=9)
n_pos = int(y_test.sum())
ax.set_xticks(range(len(steps_df)))
ax.set_xticklabels([f'{i}\nbắt {int(r["Bắt được (TP)"])}/{n_pos} | nhầm {int(r["Báo nhầm (FP)"])}'
                    for i, r in steps_df.iterrows()], fontsize=9)
ax.set(title='Test: Precision / Recall / F1 của lớp rời bỏ qua từng bước cải tiến', xlabel='', ylim=(0, 1))
plt.tight_layout(); plt.show()

**Chú thích hình:** Trên **cùng tập Test (305 khách hàng rời bỏ)**:
- Baseline Logistic Regression chỉ bắt được **113 khách hàng (Recall 0.37, F1 0.493)**.
- Chỉ đổi sang RandomForest (chưa xử lý mất cân bằng): 132 khách hàng (F1 0.562).
- Thêm `class_weight` giúp bắt nhiều nhất, **218 khách hàng (Recall 0.715)**, nhưng số báo nhầm tăng từ 33 lên **182**.
- Dời ngưỡng lên 0.61 (chọn trên Validation): mô hình cuối bắt **183 khách hàng (Recall 0.60)** và chỉ báo nhầm **85**, đạt **F1 cao nhất 0.639**.

**Kết luận:** so với baseline, mô hình cuối **bắt thêm 70 khách hàng rời bỏ (+62%)** và F1 tăng **+0.146**, đổi lại thêm 45 lần báo nhầm.
Hai bước cải tiến có vai trò khác nhau: **class weight tăng Recall, chọn ngưỡng cân lại Precision**.
Ngưỡng cuối cùng nên do bài toán kinh doanh quyết định. Nếu chi phí giữ chân rẻ, có thể dùng ngưỡng 0.5 để bắt 218/305 khách hàng.

### 4.3. Mức độ quan trọng của đặc trưng (Permutation Importance trên Test)
Xáo trộn ngẫu nhiên từng **cột gốc** và đo mức giảm PR-AUC. Cách này áp dụng cho mọi mô hình
và cho kết quả theo tên cột ban đầu (dễ giải thích hơn feature importance của cây sau one-hot).

In [ ]:
imp = permutation_importance(final_model, X_test, y_test, scoring='average_precision', n_repeats=10,
                             random_state=RANDOM_STATE, n_jobs=-1)
imp_df = pd.DataFrame({'Mức giảm PR-AUC': imp.importances_mean, 'std': imp.importances_std},
                      index=X_test.columns).sort_values('Mức giảm PR-AUC', ascending=False)
display(imp_df)

plt.figure(figsize=(9, 5))
plt.barh(imp_df.index[::-1], imp_df['Mức giảm PR-AUC'][::-1], xerr=imp_df['std'][::-1], color='#8172B3')
plt.title(f'Permutation importance — {best_name} (Test)'); plt.xlabel('Mức giảm PR-AUC khi xáo trộn cột')
plt.tight_layout(); plt.show()

**Nhận xét:** Hai feature quan trọng nhất vượt trội hẳn: **NumOfProducts** (xáo trộn làm PR-AUC giảm 0.284) và **Age** (0.243).
Tiếp theo là IsActiveMember (0.062), Geography (0.047) và Balance (0.043).
Gender, CreditScore, HasCrCard, Tenure, EstimatedSalary gần như không đóng góp (≤ 0.004).
Kết quả **khớp với EDA**: đây chính là các feature có chênh lệch tỉ lệ rời bỏ lớn nhất ở mục 1.9.

### 4.4. Phân tích lỗi (Error Analysis)
Mô hình bỏ sót (**False Negative**) và báo nhầm (**False Positive**) ở những nhóm khách hàng nào?

In [ ]:
err = X_test.copy()
err['actual'], err['prob'], err['pred'] = y_test, test_prob, test_pred
err['AgeGroup'] = pd.cut(err['Age'], [0, 30, 40, 50, 60, 100], labels=['<30', '30-39', '40-49', '50-59', '60+'])

def segment_table(col):
    g = err.groupby(col, observed=True)
    return pd.DataFrame({'Số KH': g.size(), '% rời bỏ thực tế': g['actual'].mean(),
                         'Recall': g.apply(lambda d: recall_score(d.actual, d.pred, zero_division=0)),
                         'Precision': g.apply(lambda d: precision_score(d.actual, d.pred, zero_division=0))})

fig, axes = plt.subplots(1, 4, figsize=(22, 4))
for ax, col in zip(axes, ['Geography', 'NumOfProducts', 'IsActiveMember', 'AgeGroup']):
    t = segment_table(col)
    t[['Recall', 'Precision']].plot.bar(ax=ax, rot=0, color=['#DD5145', '#4C72B0'], ylim=(0, 1.1))
    for c in ax.containers:
        ax.bar_label(c, fmt='%.2f', fontsize=8)
    ax.set(title=f'Theo {col}', xlabel='')
plt.suptitle('Recall / Precision của lớp rời bỏ theo nhóm khách hàng (Test)', y=1.03)
plt.tight_layout(); plt.show()
display(segment_table('AgeGroup'))

fn, fp = err[(err.actual == 1) & (err.pred == 0)], err[(err.actual == 0) & (err.pred == 1)]
tp = err[(err.actual == 1) & (err.pred == 1)]
print(f'False Negative: {len(fn)} | False Positive: {len(fp)}')
cols = ['Age', 'Geography', 'Balance', 'NumOfProducts', 'IsActiveMember']
num_cmp = ['Age', 'Balance', 'CreditScore', 'NumOfProducts', 'IsActiveMember']
display(pd.DataFrame({'Bỏ sót (FN)': fn[num_cmp].mean(), 'Bắt đúng (TP)': tp[num_cmp].mean(),
                      'Báo nhầm (FP)': fp[num_cmp].mean()}).rename_axis('Trung bình'))
print('Ví dụ khách hàng bị BỎ SÓT với xác suất thấp nhất:')
display(fn.sort_values('prob')[cols + ['prob']].head(5))

**Nhận xét phân tích lỗi:** trên Test có 122 khách hàng rời bỏ bị bỏ sót (FN) và 85 lần báo nhầm (FP).
- **Mô hình yếu nhất ở khách hàng trẻ:** Recall chỉ **0.13 ở nhóm < 30 tuổi** và 0.34 ở nhóm 30–39,
  so với **0.77–0.78 ở nhóm 40–59**. Khách hàng trẻ rời bỏ ít (5.6% ở nhóm < 30), nên mô hình gần như luôn đoán họ ở lại.
- **Chân dung khách hàng bị bỏ sót:** trẻ hơn (tuổi trung bình 41 so với 49 ở nhóm bắt đúng) và **hoạt động nhiều hơn**
  (47% là thành viên tích cực so với 26%), tức là "trông giống" khách hàng trung thành.
  Các ca bị bỏ sót với xác suất thấp nhất đều là khách hàng Pháp, 2 sản phẩm, đúng nhóm có tỉ lệ rời bỏ thấp nhất trong EDA.
- **Hệ quả:** với các feature hiện có, những khách hàng này gần như không thể phân biệt. Cần thêm dữ liệu hành vi
  (giao dịch gần đây, khiếu nại, tương tác với ngân hàng) mới bắt được.

### 4.5. Demo hàm suy luận
Hàm nhận thông tin **thô** của một khách hàng (có thể thiếu giá trị), tự chạy toàn bộ pipeline (tạo đặc trưng → điền thiếu →
scale → encode → mô hình) và áp dụng ngưỡng đã chọn.

In [ ]:
def predict_customer_churn(customer, model=final_model, threshold=best_threshold):
    x = pd.DataFrame([customer]).reindex(columns=X_train.columns)
    prob = float(model.predict_proba(x)[0, 1])
    return {'Xác suất rời bỏ': f'{prob:.1%}', 'Ngưỡng': threshold,
            'Kết luận': 'NGUY CƠ RỜI BỎ — nên liên hệ giữ chân' if prob >= threshold else 'Ít nguy cơ rời bỏ'}


demo_customers = {
    'KH A — 52 tuổi, Đức, 1 sản phẩm, không hoạt động': dict(CreditScore=580, Geography='Germany', Gender='Female', Age=52,
        Tenure=2, Balance=125000.0, NumOfProducts=1, HasCrCard=1, IsActiveMember=0, EstimatedSalary=85000.0),
    'KH B — 30 tuổi, Pháp, 2 sản phẩm, hoạt động': dict(CreditScore=720, Geography='France', Gender='Male', Age=30,
        Tenure=6, Balance=0.0, NumOfProducts=2, HasCrCard=1, IsActiveMember=1, EstimatedSalary=60000.0),
    'KH C — thiếu Age và Balance': dict(CreditScore=650, Geography='Spain', Gender='Female', Age=None,
        Tenure=4, Balance=None, NumOfProducts=3, HasCrCard=0, IsActiveMember=0, EstimatedSalary=None),
}
display(pd.DataFrame({k: predict_customer_churn(v) for k, v in demo_customers.items()}).T)

### 4.6. Kết luận, hạn chế và hướng phát triển


**Kết luận**
- Mô hình cuối là **RandomForest + `class_weight='balanced'` + ngưỡng 0.61**. Trên Test đạt **F1 = 0.639, Recall = 0.60, Precision = 0.68,
  ROC-AUC = 0.863**, Accuracy 0.862.
- So với baseline Logistic Regression (F1 0.493), mô hình bắt thêm **70/305 khách hàng rời bỏ**.
- F1 trên Validation (0.646) và Test (0.639) gần nhau, cho thấy mô hình **không overfit** vào Validation khi chọn ngưỡng.
- Yếu tố quyết định nhất là **số sản phẩm** và **tuổi**, sau đó là mức độ hoạt động và quốc gia.
  Gợi ý cho ngân hàng: ưu tiên chăm sóc khách hàng 40–60 tuổi, ở Đức, không hoạt động, đang dùng 1 hoặc ≥ 3 sản phẩm.

**Hạn chế**
- **Missing values do nhóm tự tạo (MCAR)**, không phản ánh cơ chế thiếu dữ liệu thực tế (thường là MAR/MNAR).
- Bộ dữ liệu phổ biến, chỉ có ảnh chụp tĩnh tại một thời điểm, không có hành vi theo thời gian. Mô hình bỏ sót nhiều khách hàng trẻ.
- Ngưỡng tối ưu theo F1 chưa tính chi phí thật của việc bỏ sót so với báo nhầm.

**Hướng phát triển**
- Chọn ngưỡng theo **chi phí kinh doanh** (ví dụ chi phí ưu đãi giữ chân so với doanh thu mất đi) thay vì F1.
- Thử SMOTE, hoặc cân chỉnh xác suất (calibration) để dùng xác suất như điểm rủi ro.
- Bổ sung dữ liệu hành vi/giao dịch theo thời gian; dùng SHAP để giải thích từng dự đoán cho nhân viên chăm sóc khách hàng.